# ROGII Wellbore Geology Prediction: submission

推論ロジックは `scripts/make_submission.py` の `run_inference` と同内容（コピー管理）。
学習済みモデル（`outputs/models/fold*.txt`）は Kaggle Dataset 経由でこの Notebook にアタッチする。
コード変更時のみこの Notebook を編集し `kaggle kernels push` で同期する。モデルだけ更新したい場合は
`kaggle datasets version` でモデル Dataset を更新するだけで良い（この Notebook の再 push は不要）。

In [ ]:
import glob
import os

import lightgbm as lgb
import numpy as np
import pandas as pd

MODEL_DIR = "/kaggle/input/rogii-models"  # TODO: 実際のモデル Dataset slug に合わせて修正
TEST_DIR = "/kaggle/input/rogii-wellbore-geology-prediction/test"  # TODO: 実際の競技入力パスに合わせて修正
GR_ROLLING_WINDOWS = [10, 30, 100]
TVT_MIN = 9245.0
TVT_MAX = 12894.0

In [ ]:
# --- src/dataset.py を移植（DTW 等の未使用部分は含めない） ---

def list_well_ids(split_dir: str) -> list[str]:
    hw_files = sorted(glob.glob(os.path.join(split_dir, "*__horizontal_well.csv")))
    return [os.path.basename(f).replace("__horizontal_well.csv", "") for f in hw_files]


def load_well(well_id: str, split_dir: str) -> pd.DataFrame:
    hw_path = os.path.join(split_dir, f"{well_id}__horizontal_well.csv")
    df = pd.read_csv(hw_path)
    df = df.sort_values("MD").reset_index(drop=True)
    df["well_id"] = well_id
    df["row_index"] = df.index
    return attach_prefix_tail(df)


def load_typewell(well_id: str, split_dir: str) -> pd.DataFrame:
    tw_path = os.path.join(split_dir, f"{well_id}__typewell.csv")
    return pd.read_csv(tw_path).sort_values("TVT").reset_index(drop=True)


def attach_prefix_tail(df: pd.DataFrame) -> pd.DataFrame:
    known = df["TVT_input"].notna()
    transitions = known.astype(int).diff().fillna(0)
    n_breaks = (transitions == -1).sum()
    assert n_breaks <= 1, f"prefix->tail の遷移が複数回検出された (n_breaks={n_breaks})"

    df = df.copy()
    df["is_tail"] = ~known
    anchor_row = df.loc[known].iloc[-1] if known.any() else df.iloc[0]
    df["last_known_TVT"] = anchor_row["TVT_input"] if known.any() else float("nan")
    df["delta_MD"] = df["MD"] - anchor_row["MD"]
    df["delta_X"] = df["X"] - anchor_row["X"]
    df["delta_Y"] = df["Y"] - anchor_row["Y"]
    df["delta_Z"] = df["Z"] - anchor_row["Z"]
    df["neg_Z"] = -df["Z"]
    return df

In [ ]:
# --- src/features.py を移植（DTW関数は exp001 で未使用と判明したため移植しない） ---

def calibrate_gr(hw: pd.DataFrame, tw: pd.DataFrame) -> pd.Series:
    prefix_mask = ~hw["is_tail"]
    hw_gr_filled = hw["GR"].interpolate(limit_direction="both")

    prefix_tvt_range = hw.loc[prefix_mask, "TVT_input"]
    if prefix_mask.sum() == 0 or prefix_tvt_range.isna().all():
        return hw_gr_filled

    tvt_lo, tvt_hi = prefix_tvt_range.min(), prefix_tvt_range.max()
    tw_in_range = tw[(tw["TVT"] >= tvt_lo) & (tw["TVT"] <= tvt_hi)]
    if len(tw_in_range) == 0:
        return hw_gr_filled

    hw_prefix_mean = hw_gr_filled[prefix_mask].mean()
    tw_mean = tw_in_range["GR"].mean()
    shift = tw_mean - hw_prefix_mean
    return hw_gr_filled + shift


def rolling_gr_features(hw: pd.DataFrame, gr_cal: pd.Series, windows: list[int]) -> pd.DataFrame:
    out = {}
    out["gr_cal"] = gr_cal
    out["gr_gradient"] = gr_cal.diff().fillna(0.0)
    for w in windows:
        out[f"gr_roll_mean_{w}"] = gr_cal.rolling(w, min_periods=1).mean()
        out[f"gr_roll_std_{w}"] = gr_cal.rolling(w, min_periods=1).std().fillna(0.0)
    return pd.DataFrame(out, index=hw.index)


def build_feature_frame(hw: pd.DataFrame, tw: pd.DataFrame, gr_rolling_windows: list[int]) -> pd.DataFrame:
    gr_cal = calibrate_gr(hw, tw)
    rolling_feat = rolling_gr_features(hw, gr_cal, gr_rolling_windows)
    feat = pd.concat(
        [
            hw[["well_id", "row_index", "MD", "X", "Y", "Z", "delta_MD",
                "delta_X", "delta_Y", "delta_Z", "neg_Z", "is_tail", "last_known_TVT"]],
            rolling_feat,
        ],
        axis=1,
    )
    if "TVT" in hw.columns:
        feat["TVT"] = hw["TVT"]
        feat["resid"] = feat["TVT"] - feat["last_known_TVT"]
    return feat

In [ ]:
# --- scripts/make_submission.py の run_inference をそのままコピー ---

NON_FEATURE_COLS = ["well_id", "row_index", "is_tail", "TVT", "resid", "tvt_diff_abs"]


def run_inference(models, test_dir, gr_rolling_windows, tvt_min, tvt_max) -> pd.DataFrame:
    well_ids = list_well_ids(test_dir)
    rows = []
    for well_id in well_ids:
        hw = load_well(well_id, test_dir)
        tw = load_typewell(well_id, test_dir)
        feat = build_feature_frame(hw, tw, gr_rolling_windows)

        tail_feat = feat[feat["is_tail"]].copy()
        feature_cols = [c for c in feat.columns if c not in NON_FEATURE_COLS]

        preds = np.mean([m.predict(tail_feat[feature_cols]) for m in models], axis=0)
        pred_tvt = tail_feat["last_known_TVT"].to_numpy() + preds
        pred_tvt = np.clip(pred_tvt, tvt_min, tvt_max)

        ids = [f"{well_id}_{int(r)}" for r in tail_feat["row_index"]]
        rows.append(pd.DataFrame({"id": ids, "tvt": pred_tvt}))

    return pd.concat(rows, ignore_index=True)

In [ ]:
model_paths = sorted(glob.glob(f"{MODEL_DIR}/fold*.txt"))
assert model_paths, f"{MODEL_DIR} にモデルファイルが見つかりません"
models = [lgb.Booster(model_file=p) for p in model_paths]

submission = run_inference(models, TEST_DIR, GR_ROLLING_WINDOWS, TVT_MIN, TVT_MAX)
submission.to_csv("submission.csv", index=False)
print(f"saved submission.csv ({len(submission)} rows)")
submission.head()